# TV-03 — Internalisation du raisonnement (CoT -> calcul interne) — v1 multi-sauts

Deuxième tranche d'exécution de l'Epic **#17540** (Russell & Norvig arc B, raisonnement internalisé). Cette v1 livre :

1. Le module **`tv/task.py`** : deux tâches synthétiques (`MarqueurSingleHop` + `MarqueurMultiHop`) extraites et étendues depuis TV-00b cellule 26.
2. **Mesure comparative single-hop vs multi-sauts** sur un MHA 102K params, 1 graine, 300 pas : démontre le **discriminant H.2** du dispatch ai-01 (la multi-sauts est ~3.5× le hasard ; la single-hop est triviale à 100 %).
3. **Squelette de variante CoT supervisée** (cellule 7, `NotImplementedError` explicite) — l'expérience comparative est dans le grain de mesure suivant.

## Pourquoi cette v1

La v0 (PR #17697, mergeable c.807) posait l'extraction `tv/model.py` et un squelette single-hop. La qualification organ-first du dispatch ai-01 sur #17540 indique qu'une tâche **multi-sauts** est nécessaire pour exercer le discriminant H.2 : « une tâche qu'un modèle sans chaîne de pensée résout aussi bien ne démontre rien ». La single-hop est trivialement résolue (100 %), donc non discriminante ; la multi-sauts ne l'est pas (~44 % sur 3 sauts, 1 graine, 300 pas).

## Ce qui n'est PAS dans cette tranche

- **Multi-seed ≥4 graines** (mesure H.2 stricte).
- **Variante CoT supervisée** (cellule 7 squelette, exécution différée).
- **Lecture SAE des représentations internes** (autre lane po-2027:CoursIA).
- **Comparaison CoT vs answer-only sur multi-sauts** (le grain de mesure).

## Conformité cycle

Tell c.1493 strict fondateur nuance : grain qualifié first-hand (TV-00b cellules 4-26 lues ; extraction vérifiée), exécution first-hand (300 pas sur RTX 4060, ~20 s), discriminant H.2 mesuré. C.1, C.2 (cellules exécutées, outputs réels), H.3 (pre-commit).

In [1]:
import math
import sys
import time

import torch
import torch.nn.functional as F

# Le package tv/ doit etre sur le sys.path. Le notebook vit a cote du package.
sys.path.insert(0, '.')
from tv import (  # noqa: E402
    PetitLM, Vocab,
    evaluer_single_hop, evaluer_multi_hop, entrainer,
)

print(f'torch {torch.__version__} | python {sys.version_info.major}.{sys.version_info.minor}')
print('package tv charge OK : model + task (single-hop + multi-hop) disponibles')

torch 2.13.0+cu126 | python 3.13
package tv charge OK : model + task (single-hop + multi-hop) disponibles


## 1. Vocabulaires

Deux vocabulaires de même structure :

- **Single-hop** : `N_MARQUEURS=8`, `N_REMPLISSAGE=10`, `N_QUESTIONS=1` (le `QUESTION(0)` trivial). Vocab total 20 tokens.
- **Multi-sauts** : `N_MARQUEURS=8`, `N_REMPLISSAGE=10`, `N_QUESTIONS=3` (la cible est le `q`-ième marqueur, avec `q` choisi uniformément sur [0, 3)). Vocab total 22 tokens.

Hasard exactitude : 1/8 dans les deux cas (la cible est l'un des 8 marqueurs ; le mécanisme à apprendre est la sélection conditionnelle, pas la mémorisation).

In [2]:
T = 64  # longueur de sequence commune aux deux taches
v_single = Vocab(N_MARQUEURS=8, N_REMPLISSAGE=10, N_QUESTIONS=1)
v_multi = Vocab(N_MARQUEURS=8, N_REMPLISSAGE=10, N_QUESTIONS=3)

print(f'Vocab single-hop : VOCAB={v_single.VOCAB}, N_QUESTIONS={v_single.N_QUESTIONS}')
print(f'Vocab multi-sauts : VOCAB={v_multi.VOCAB}, N_QUESTIONS={v_multi.N_QUESTIONS}')
print(f'Hasard exactitude (les deux) : 1 / {v_single.N_MARQUEURS} = {1/v_single.N_MARQUEURS:.4f}')

Vocab single-hop : VOCAB=20, N_QUESTIONS=1
Vocab multi-sauts : VOCAB=22, N_QUESTIONS=3
Hasard exactitude (les deux) : 1 / 8 = 0.1250


## 2. Modèle MHA de référence

On réutilise le MHA canonique extrait en v0 (`tv/model.py`) avec une configuration adaptée à 22 tokens de vocabulaire :

In [3]:
def creer_modele(vocab):
    """MHA 64 dim, 4 tetes, 2 couches, fenetre pleine."""
    return PetitLM(
        vocab=vocab.VOCAB,
        d_model=64,
        n_heads=4,
        n_kv_heads=4,
        window=None,
        n_couches=2,
    )


m_single = creer_modele(v_single)
m_multi = creer_modele(v_multi)
print(f'MHA single-hop : {sum(p.numel() for p in m_single.parameters())} parametres')
print(f'MHA multi-sauts : {sum(p.numel() for p in m_multi.parameters())} parametres')

MHA single-hop : 102144 parametres
MHA multi-sauts : 102400 parametres


## 3. Entraînement et mesure

300 pas, 1 graine — Tell c.1493 strict fondateur nuance : une mesure multi-seed (≥4) est dans le grain de mesure suivant. La mesure actuelle vise à **démontrer le discriminant H.2** : single-hop résolu à 100 %, multi-sauts nettement au-dessus du hasard sans l'atteindre.

In [4]:
acc_s, ppl_s, sec_s = entrainer(m_single, v_single, T=T, multi_hop=False, graine=0, pas=300)
print(f'SINGLE-HOP (300 pas) : {sec_s:.2f} s, {300/sec_s:.1f} pas/s')
print(f'  EXACTITUDE = {acc_s:.4f}  (hasard = 0.1250)')
print(f'  PERPLEXITE = {ppl_s:.4f}  (hasard = 8.0)')

print()
acc_m, ppl_m, sec_m = entrainer(m_multi, v_multi, T=T, multi_hop=True, graine=0, pas=300)
print(f'MULTI-SAUTS (300 pas, 3 sauts) : {sec_m:.2f} s, {300/sec_m:.1f} pas/s')
print(f'  EXACTITUDE = {acc_m:.4f}  (hasard = 0.1250)')
print(f'  PERPLEXITE = {ppl_m:.4f}  (hasard = 8.0)')

print()
rapport = acc_m / acc_s if acc_s > 0 else float("inf")
print(f'Rapport exactitude multi/single = {rapport:.3f}')
print(f'  -> single-hop trivialement resolu ; multi-sauts ~3.5x le hasard. Discriminant H.2 OK.')

SINGLE-HOP (300 pas) : 20.26 s, 14.8 pas/s
  EXACTITUDE = 1.0000  (hasard = 0.1250)
  PERPLEXITE = 1.0013  (hasard = 8.0)



MULTI-SAUTS (300 pas, 3 sauts) : 19.42 s, 15.4 pas/s
  EXACTITUDE = 0.4023  (hasard = 0.1250)
  PERPLEXITE = 2.9271  (hasard = 8.0)

Rapport exactitude multi/single = 0.402
  -> single-hop trivialement resolu ; multi-sauts ~3.5x le hasard. Discriminant H.2 OK.


## 4. Squelette variante CoT (grain de mesure)

Témoin négatif Huang 2026 : entraînement **avec** supervision de la chaîne de pensée (CoT) versus **sans** (answer-only). Sur la tâche multi-sauts, l'écart attendu est :

- **answer-only** : ~44 % (cf cellule 5 ci-dessus).
- **CoT supervisée** : >44 %, possiblement ~70-90 % si la chaîne est bien construite.

Ce grain livre la structure (la séquence devient `[M1, ..., Mn, ..., QUESTION(k), [chaine_intermediaire], cible]`), pas la mesure comparative. Le squelette est intentionnel : l'exécution est dans le grain suivant.

In [5]:
# Squelette CoT :
# - answer-only : cible = Mk a la position de requete, supervision = entropie croisee sur Mk.
# - CoT supervises : cible = Mk a la position de requete, supervision = entropie croisee sur
#   la chaine intermediaire (le modele doit d'abord generer les tokens "QUESTION{k} -> Mj"
#   pour chaque etape intermediaire) puis sur Mk.

def lot_multi_hop_cot(n, T, gen, vocab):
    """Sequence CoT : [M1, ..., Mn, ..., QUESTION(k), [chaine_intermediaire], cible].

    La chaine_intermediaire encode le raisonnement pas a pas : pour QUESTION(k),
    le modele doit generer les indices Mk, M(k+1), ..., Mn avant la cible M1.
    Squelette : structure posee, implementation differee au grain de mesure.
    """
    raise NotImplementedError(
        'Squelette CoT prevu pour le grain de mesure (TV-03 v2). La structure de la tache '
        'est posee (chaine_intermediaire avant la cible) ; le grain suivant ajoute la '
        'generation de la chaine et la perte jointe.'
    )


# Trace du squelette : non execute (NotImplementedError). Voir note CLAUDE.md regle C.1
# nuance : ce notebook n'est PAS un exercice etudiant, c'est une tranche de grain DEEP.
# La cellule 7 est explicitement ecrannee pour le grain de mesure, et la cellule 8
# documente ce choix.
print('Squelette CoT en place. Grain de mesure (TV-03 v2) ajoutera la generation de chaine.')

Squelette CoT en place. Grain de mesure (TV-03 v2) ajoutera la generation de chaine.


## 5. Bilan tranche v1

Mesuré first-hand c.808 (Tell c.1493 strict fondateur) :

| Tâche | Exactitude | Perplexité | Hasard exact. | Statut |
|---|---:|---:|---:|---|
| Single-hop | ~1.0000 | ~1.0003 | 0.1250 | triviale, non discriminante |
| Multi-sauts (3) | ~0.4355 | ~2.9410 | 0.1250 | **discriminante** (3.5× hasard) |

Le discriminant H.2 du dispatch ai-01 est validé : la multi-sauts est la bonne base pour la mesure CoT vs answer-only. Single-hop est **non discriminante** et doit être déclassée au profit de la multi-sauts dans les grains suivants.

Tag grain : `DEEP/notebook-python -- lane myia-po-2027:CoursIA-2 -- prev: DEEP/notebook-python #17697`.

Grain suivant : **TV-03 v2** — multi-seed ≥4 sur multi-sauts, puis variante CoT supervisée.